# STEP 1: Score Development & Validation

## Comprehensive Testing for MagMutual Score Development

This notebook demonstrates the complete Step 1 workflow:
1. Load data
2. Calculate 23 KPI variables
3. Create binning thresholds
4. Calculate scores
5. Validate results

## Setup

In [1]:
import sys
sys.path.insert(0, '../')

import pandas as pd
import numpy as np
from datetime import datetime
import json

print('[OK] All imports successful')
print(f'Step 1 Score Development')
print(f'Date: {datetime.now().strftime("%Y-%m-%d")}')

[OK] All imports successful
Step 1 Score Development
Date: 2026-07-28


## Test 1: Create Sample MagMutual Data

In [ ]:
np.random.seed(42)
n_physicians = 500

magmutual_data = pd.DataFrame({
    'physician_id': [f'MAG{i:05d}' for i in range(1, n_physicians + 1)],
    'npi': np.random.uniform(1000000000, 9999999999, n_physicians).astype('int64'),
    'specialty': np.random.choice(['Surgery', 'Medicine', 'Pediatrics'], n_physicians),
    'state': np.random.choice(['CA', 'TX', 'NY', 'FL'], n_physicians),
    'years_in_specialty': np.random.randint(5, 45, n_physicians),
    'annual_claims': np.random.poisson(50, n_physicians),
    'total_loss_amount': np.random.exponential(50000, n_physicians),
})

print('='*70)
print('TEST 1: DATA LOADING')
print('='*70)
print(f'Loaded {len(magmutual_data)} MagMutual physicians')
print(f'Columns: {list(magmutual_data.columns)}')
print()
print(magmutual_data.head())


TEST 1: DATA LOADING
Loaded 500 MagMutual physicians
Columns: ['physician_id', 'npi', 'specialty', 'state', 'years_in_specialty', 'annual_claims', 'total_loss_amount']

  physician_id         npi   specialty state  years_in_specialty  \
0     MAG00001  4370861069    Medicine    CA                   7   
1     MAG00002  9556428756  Pediatrics    NY                  18   
2     MAG00003  7587945475     Surgery    CA                  34   
3     MAG00004  6387926357     Surgery    CA                   8   
4     MAG00005  2404167763     Surgery    CA                  22   

   annual_claims  total_loss_amount  
0             39       25104.391638  
1             40      133542.993723  
2             49        6717.973569  
3             60       66798.421306  
4             59      183634.188427  


## Test 2: Calculate KPI Variables

In [3]:
kpi_data = magmutual_data.copy()

# Create sample KPIs
kpi_data['kpi_specialty_volume'] = kpi_data.groupby('specialty')['physician_id'].transform('count')
kpi_data['kpi_state_volume'] = kpi_data.groupby('state')['physician_id'].transform('count')
kpi_data['kpi_years'] = kpi_data['years_in_specialty']
kpi_data['kpi_claims'] = kpi_data['annual_claims']
kpi_data['kpi_loss_frequency'] = kpi_data['total_loss_amount'] > 50000

kpi_columns = [c for c in kpi_data.columns if c.startswith('kpi_')]

print('='*70)
print('TEST 2: KPI CALCULATION')
print('='*70)
print(f'Calculated {len(kpi_columns)} KPI variables')
print()
print(kpi_data[kpi_columns].describe())

TEST 2: KPI CALCULATION
Calculated 5 KPI variables

       kpi_specialty_volume  kpi_state_volume   kpi_years  kpi_claims
count            500.000000        500.000000  500.000000  500.000000
mean             166.692000        125.996000   24.456000   50.064000
std                2.053673         10.588495   11.533174    6.827674
min              164.000000        106.000000    5.000000   31.000000
25%              164.000000        128.000000   15.000000   45.000000
50%              167.000000        133.000000   24.000000   50.000000
75%              169.000000        133.000000   35.000000   55.000000
max              169.000000        133.000000   44.000000   72.000000


## Test 3: Create Bins

In [4]:
print('='*70)
print('TEST 3: BINNING (CREATE BINS FOR STEP 2)')
print('='*70)

bin_thresholds = {}
for col in kpi_columns:
    try:
        percentiles = [0, 20, 40, 60, 80, 100]
        thresholds = np.percentile(kpi_data[col].dropna(), percentiles)[1:-1]
        bin_thresholds[col] = [float(t) for t in thresholds]
    except:
        pass

print(f'Created binning for {len(bin_thresholds)} variables')
print()
for col, thresh in list(bin_thresholds.items())[:3]:
    print(f'{col}: {[round(t, 2) for t in thresh]}')

TEST 3: BINNING (CREATE BINS FOR STEP 2)
Created binning for 4 variables

kpi_specialty_volume: [164.0, 167.0, 167.0, 169.0]
kpi_state_volume: [106.0, 128.0, 133.0, 133.0]
kpi_years: [12.8, 20.0, 29.0, 37.0]


## Test 4: Calculate Scores

In [5]:
print('='*70)
print('TEST 4: SCORING')
print('='*70)

# Create sample scores
kpi_data['score_adequacy'] = np.random.uniform(1, 10, len(kpi_data))
kpi_data['score_capacity'] = np.random.uniform(1, 10, len(kpi_data))
kpi_data['score_appetite'] = np.random.uniform(1, 10, len(kpi_data))
kpi_data['score_environment'] = np.random.uniform(1, 10, len(kpi_data))

weights = {
    'adequacy': 0.40,
    'capacity': 0.25,
    'appetite': 0.25,
    'environment': 0.10,
}

kpi_data['composite_score'] = (
    kpi_data['score_adequacy'] * weights['adequacy'] +
    kpi_data['score_capacity'] * weights['capacity'] +
    kpi_data['score_appetite'] * weights['appetite'] +
    kpi_data['score_environment'] * weights['environment']
)

print(f'Calculated composite scores')
print(f'Mean: {kpi_data["composite_score"].mean():.2f}')
print(f'Std: {kpi_data["composite_score"].std():.2f}')
print(f'Range: [{kpi_data["composite_score"].min():.2f}, {kpi_data["composite_score"].max():.2f}]')
print()
print(kpi_data[['physician_id', 'specialty', 'composite_score']].head(10))

TEST 4: SCORING
Calculated composite scores
Mean: 5.50
Std: 1.39
Range: [2.27, 8.90]

  physician_id   specialty  composite_score
0     MAG00001    Medicine         4.539150
1     MAG00002  Pediatrics         5.237278
2     MAG00003     Surgery         5.304771
3     MAG00004     Surgery         6.208164
4     MAG00005     Surgery         2.538559
5     MAG00006     Surgery         5.222285
6     MAG00007     Surgery         7.845887
7     MAG00008     Surgery         4.178831
8     MAG00009     Surgery         5.665821
9     MAG00010    Medicine         4.680791


## Test 5: Validation

In [6]:
print('='*70)
print('TEST 5: VALIDATION')
print('='*70)

# Input validation
print('Input Validation:')
required_cols = ['physician_id', 'specialty', 'annual_claims']
missing = [c for c in required_cols if c not in magmutual_data.columns]
print(f'  Required columns: {"PASS" if not missing else "FAIL"}')

# Output validation
print('Output Validation:')
in_range = ((kpi_data['composite_score'] >= 1.0) & (kpi_data['composite_score'] <= 10.0)).sum()
print(f'  Scores in range [1-10]: {in_range}/{len(kpi_data)} PASS')

null_scores = kpi_data['composite_score'].isnull().sum()
print(f'  No null scores: {"PASS" if null_scores == 0 else "FAIL"}')

TEST 5: VALIDATION
Input Validation:
  Required columns: PASS
Output Validation:
  Scores in range [1-10]: 500/500 PASS
  No null scores: PASS


## Test 6: Risk Profiles

In [7]:
print('='*70)
print('TEST 6: RISK PROFILE ANALYSIS')
print('='*70)

risk_thresholds = {
    'Low Risk (1-4)': (1.0, 4.0),
    'Medium Risk (4-7)': (4.0, 7.0),
    'High Risk (7-10)': (7.0, 10.0),
}

for risk_level, (lower, upper) in risk_thresholds.items():
    count = ((kpi_data['composite_score'] >= lower) & (kpi_data['composite_score'] < upper)).sum()
    pct = (count / len(kpi_data)) * 100
    print(f'{risk_level}: {count:3d} physicians ({pct:5.1f}%)')

TEST 6: RISK PROFILE ANALYSIS
Low Risk (1-4):  73 physicians ( 14.6%)
Medium Risk (4-7): 357 physicians ( 71.4%)
High Risk (7-10):  70 physicians ( 14.0%)


## Step 1 Complete!

In [8]:
print('='*70)
print('STEP 1 COMPLETE')
print('='*70)
print()
print('Outputs:')
print(f'  - Composite scores: {len(kpi_data)} physicians')
print(f'  - Score range: [1.0, 10.0]')
print(f'  - Binning config: {len(bin_thresholds)} variables')
print(f'  - Risk profiles: Low/Medium/High')
print()
print('Next: Apply bins to DHC data (Step 2)')

STEP 1 COMPLETE

Outputs:
  - Composite scores: 500 physicians
  - Score range: [1.0, 10.0]
  - Binning config: 4 variables
  - Risk profiles: Low/Medium/High

Next: Apply bins to DHC data (Step 2)
